# 01 generate flights

Design decisions: PX4 main built for SITL and cached as a tarball in `data/px4_cache`; the `sensor_gps_sim` patch is idempotent and re-applied before every build; one fresh PX4 daemon per flight; run name selects the output folder under `data/sih`.

In [ ]:
# ============================================================
# BOOTSTRAP  (top of every notebook in this project)
# ============================================================
import os, sys, subprocess, shutil
from pathlib import Path

PROJECT   = "UAV_GNSS"
REPO_NAME = "uav-gnss-triage"

DRIVE_MOUNT = Path("/content/drive")
DRIVE_ROOT  = DRIVE_MOUNT / "MyDrive" / f"{PROJECT}_Research"
REPO_DIR    = DRIVE_ROOT / REPO_NAME

if not (DRIVE_MOUNT / "MyDrive").exists():
    from google.colab import drive
    drive.mount(str(DRIVE_MOUNT))

for dotfile in (".gitconfig", ".git-credentials"):
    src = DRIVE_ROOT / dotfile
    if src.exists():
        shutil.copy(src, Path.home() / dotfile)
cred = Path.home() / ".git-credentials"
if cred.exists():
    os.chmod(cred, 0o600)
subprocess.run(["git", "config", "--global", "credential.helper", "store"], check=False)

if REPO_DIR.exists():
    os.chdir(REPO_DIR)
    if str(REPO_DIR / "src") not in sys.path:
        sys.path.insert(0, str(REPO_DIR / "src"))
    import paths as P
print("CWD:", os.getcwd(), "| credentials:", cred.exists())


In [ ]:
# ---- PX4 build tree: restore from cache or build from source, then apply the patch and rebuild ----
RUN = "sih_flights_v2"
subprocess.run(["pip", "install", "-q", "mavsdk", "pyulog"], check=True)
cache = P.PX4_CACHE / "px4_autopilot.tgz"
P.PX4_CACHE.mkdir(parents=True, exist_ok=True)

def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, text=True, capture_output=True)
    print(r.stdout[-1500:], r.stderr[-1500:]); return r.returncode

if not P.PX4_SRC.exists():
    if cache.exists():
        sh(f"tar -C /content -xzf {cache}")
    else:
        sh(f"git clone --recursive https://github.com/PX4/PX4-Autopilot.git {P.PX4_SRC}")
        sh("bash Tools/setup/ubuntu.sh --no-nuttx --no-sim-tools", cwd=P.PX4_SRC)
sh(f"{sys.executable} {P.SRC / 'sih_spoof_patch.py'} {P.PX4_SRC}")
rc = sh("make px4_sitl_default 2>&1 | tail -3", cwd=P.PX4_SRC)
assert rc == 0, "PX4 build failed"
sh(f"tar -C /content -czf {cache} PX4-Autopilot")
print("firmware:", subprocess.run(["git", "-C", str(P.PX4_SRC), "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())


In [ ]:
# ---- PX4 build tree: restore from cache or build from source, then apply the patch and rebuild ----
RUN = "sih_flights_v2"
subprocess.run(["pip", "install", "-q", "mavsdk", "pyulog"], check=True)
cache = P.PX4_CACHE / "px4_autopilot.tgz"
P.PX4_CACHE.mkdir(parents=True, exist_ok=True)

def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, text=True, capture_output=True)
    print(r.stdout[-1500:], r.stderr[-1500:]); return r.returncode

if not P.PX4_SRC.exists():
    if cache.exists():
        sh(f"tar -C /content -xzf {cache}")
    else:
        sh(f"git clone --recursive https://github.com/PX4/PX4-Autopilot.git {P.PX4_SRC}")
if shutil.which("ninja") is None:          # fresh runtime: toolchain and Python deps are not persisted
    sh("bash Tools/setup/ubuntu.sh --no-nuttx --no-sim-tools 2>&1 | tail -3", cwd=P.PX4_SRC)
sh(f"{sys.executable} {P.SRC / 'sih_spoof_patch.py'} {P.PX4_SRC}")
rc = sh("make px4_sitl_default > /tmp/px4_build.log 2>&1; rc=$?; tail -3 /tmp/px4_build.log; exit $rc", cwd=P.PX4_SRC)
assert rc == 0, "PX4 build failed"
sh(f"tar -C /content -czf {cache} PX4-Autopilot")
print("firmware:", subprocess.run(["git", "-C", str(P.PX4_SRC), "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())

In [ ]:
# ---- redo flights that failed in a previous run: drop their stubs, regenerate with identical seeds ----
import json, shutil
run = P.SIH_RUNS / RUN
lines = [l for l in open(run / "manifest.jsonl") if l.strip()]
keep, drop = [], []
for l in lines:
    (keep if json.loads(l)["ok"] else drop).append(json.loads(l))
shutil.copy(run / "manifest.jsonl", run / "manifest_before_redo.jsonl")
with open(run / "manifest.jsonl", "w") as f:
    for r in keep:
        f.write(json.dumps(r) + "\n")
for r in drop:
    for ext in (".ulg", ".json", ".px4.txt"):
        p = run / f"{r['flight_id']}{ext}"
        if p.exists():
            p.unlink()
print("kept", len(keep), "| dropped for regeneration", len(drop))

proc = subprocess.Popen([sys.executable, str(P.SRC / "sih_generate_flights.py"), "--px4", str(P.PX4_SRC),
                         "--out", str(run), "--n_per_family", "60", "--speed", "8"],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    if not line.startswith("skip (done)"):
        print(line, end="")
print("exit code:", proc.wait())

In [ ]:
# ---- generate: resume-safe, skips flights already in the manifest ----
out = P.SIH_RUNS / RUN
proc = subprocess.Popen([sys.executable, str(P.SRC / "sih_generate_flights.py"), "--px4", str(P.PX4_SRC),
                         "--out", str(out), "--n_per_family", "60", "--speed", "8"],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
print("exit code:", proc.wait())


In [ ]:
# ---- manifest quality check ----
import json, pandas as pd
rows = [json.loads(l) for l in open(P.SIH_RUNS / RUN / "manifest.jsonl") if l.strip()]
df = pd.json_normalize(rows)
print("flights:", len(df), "| ok:", int(df["ok"].sum()), "| with notes:", int((df["notes"].str.len() > 0).sum()))
print(df.groupby(["family", "subtype"]).size().to_string())
print(pd.Series([n.split(":")[0].split(" on attempt")[0] for ns in df["notes"] for n in ns]).value_counts().to_string())
print(df.groupby("family")["verify.max_gps_vs_truth_m"].describe()[["count", "min", "50%", "max"]].round(1).to_string())
print(df.groupby("family")["verify.duration_s"].describe()[["min", "50%", "max"]].round(0).to_string())
print("gps_degrade last30s:", df[df.family == "gps_degrade"]["verify.div_last30s_max_m"].describe()[["50%", "max"]].round(1).to_dict())


In [ ]:
import json, pandas as pd, re
run = P.SIH_RUNS / "sih_flights_v2"
rows = [json.loads(l) for l in open(run / "manifest.jsonl") if l.strip()]
df = pd.json_normalize(rows)
bad = df[~df.ok]
print("failed flights:", len(bad), "| by family:", bad.family.value_counts().to_dict())
print("\nfull notes of the first 5 failures:")
for _, r in bad.head(5).iterrows():
    print(r.flight_id, r.notes)
print("\nnuisance, failed vs ok (median):")
for c in ["nuisance.SIM_GPS_NOISE_T", "nuisance.SIM_GPS_NOISE_P", "nuisance.SIM_GPS_NOISE_J", "nuisance.SIM_GPS_NOISEV_K"]:
    print(f"  {c:28s} failed={bad[c].median():.2f}  ok={df[df.ok][c].median():.2f}")
print("\nPX4 preflight messages in failed flights:")
hits = {}
for fid in bad.flight_id:
    txt = (run / f"{fid}.px4.txt")
    if txt.exists():
        for line in txt.read_text(errors="ignore").splitlines():
            if "Preflight" in line or "Fail" in line or "reject" in line.lower():
                key = re.sub(r"\d+(\.\d+)?", "#", line.strip())
                hits[key] = hits.get(key, 0) + 1
for k, v in sorted(hits.items(), key=lambda x: -x[1])[:10]:
    print(f"  {v:3d}  {k}")